<div style="background: linear-gradient(135deg, #050505 0%, #0d1117 30%, #0a2a1a 70%, #0d1f3a 100%);padding: 60px 40px; border-radius: 18px; text-align: center; margin-bottom: 10px; border: 1px solid #2ea04344;"><h1 style="color: #2ea043; font-family: 'Courier New', monospace; font-size: 2.9em;letter-spacing: 5px; margin: 0 0 12px 0; text-shadow: 0 0 40px #2ea04355;">🖼️ OpenCV DEVAM</h1><h2 style="color: #a8dadc; font-family: 'Courier New', monospace;font-size: 1.35em; margin: 0 0 18px 0; font-weight: 300;">5. Hafta — Görüntü Üretme, Kaydetme & Video İşleme</h2><p style="color: #6b7280; font-family: 'Courier New', monospace; font-size: 0.85em; margin: 0;">NumPy ile Görüntü Üretme &bull; imwrite Bayrakları &bull; Video Okuma &bull; Piksel Manipülasyonu &bull; Örüntü Oluşturma</p></div>

---
# 🔬 BÖLÜM 1: Dijital Görüntünün İç Yapısı — Derinlemesine

Bu haftanın ana teması: **görüntüleri sıfırdan üretmek**. Bunun için önce dijital görüntünün tam olarak ne olduğunu anlamamız gerekiyor.

## 1.1 Bir Görüntü = Sayılar Matrisi

```
Gri tonlamalı (256×256):       Renkli BGR (256×256):
┌─────────────────────┐        ┌───────────────────────────┐
│  0   0   0 … 255   │        │ [255,0,0] [0,255,0] …     │  ← Mavi kanal
│  0  50 100 … 200   │        │ [0,0,255] [128,0,128] …   │  ← Yeşil kanal
│  …   …   … …  …   │        │  …         …               │  ← Kırmızı kanal
└─────────────────────┘        └───────────────────────────┘
 shape=(256,256)   dtype=uint8   shape=(256,256,3)  dtype=uint8
  65,536 byte RAM                  196,608 byte RAM
```

Her piksel bir **uint8** (0–255 arası işaretsiz tam sayı). Bu seçim rastgele değil: insan gözü yaklaşık **256 parlaklık seviyesini** ayırt edebilir.

> **Önemli soru:** Neden 8 bit, 16 bit değil? — 8 bit = 256 seviye insan algısı için yeterli ve depolama/işlem açısından verimlidir. Tıbbi görüntülemede (MRI, CT) 16 bit (65,536 seviye) kullanılır.

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Piksel Kavramının Tarihi:</b><br><br>"Pixel" kelimesi ilk kez 1965'te NASA'nın Jet Propulsion Laboratory'sinde kullanıldı. Fredric Billingsley, uzay araçlarından gelen görüntü elementlerini adlandırmak için "picture element" → "pix-el" kısaltmasını türetti.<br><br>İlk dijital görüntü işleme uygulaması: 1957'de Russell Kirsch, oğlunun fotoğrafını 176×176 piksel çözünürlükte taradı. Bu görüntü bugün hâlâ "dünyanın en etkili fotoğraflarından biri" olarak kabul edilir — çünkü dijital görüntüleme çağını başlattı.<br><br><b>Renkli dijital görüntü:</b> İlk renkli dijital fotoğraf 1972'de NASA tarafından Skylab astronotlarının çektiği Dünya görüntüleriydi.</span></div>

## 1.2 dtype Seçiminin Önemi

| dtype | Aralık | Boyut | Kullanım |
|-------|--------|-------|----------|
| `uint8` | 0–255 | 1 byte | Standart görüntüler (OpenCV varsayılanı) |
| `uint16` | 0–65,535 | 2 byte | Tıbbi görüntüler, RAW fotoğraf |
| `float32` | −∞ – +∞ | 4 byte | Derin öğrenme tensörleri, ara işlemler |
| `float64` | −∞ – +∞ | 8 byte | Hassas bilimsel hesaplama |
| `int16` | −32,768 – 32,767 | 2 byte | Laplacian fark görüntüleri (negatif değer) |

⚠️ **Taşma tuzağı:** `uint8` ile `200 + 100 = 300` → `300 % 256 = 44` (sessiz taşma!). Aritmetik işlemler için önce `float32`'ye çevirin.

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>uint8 Taşması — En Sinsi Bug:</b><br><br><code>import numpy as np<br>a = np.array([200], dtype=np.uint8)<br>b = np.array([100], dtype=np.uint8)<br>print(a + b)  # [44] — beklenen 300 değil!</code><br><br>Bu, görüntü işlemede yüzlerce saatlik hata ayıklamaya yol açan bir tuzaktır. Çözüm: <code>cv.add(a, b)</code> — OpenCV'nin add() fonksiyonu otomatik saturasyon yapar: sonuç 255'i geçemez, "çevresel" (wrap-around) taşma olmaz. <code>cv.subtract()</code> da aynı şekilde 0'ın altına düşmez.</span></div>

---
# 📖 BÖLÜM 2: Görüntü Okuma — Farklı Modlar

## 2.1 Üç Farklı Okuma Modu

In [ ]:
import cv2 as cv
import numpy as np

# Üç farklı okuma modu
renkli = cv.imread("resim/manzara.jpg", cv.IMREAD_UNCHANGED)
saydam = cv.imread("resim/bilgisayar.png", cv.IMREAD_UNCHANGED)
gri    = cv.imread("resim/manzara.jpg", cv.IMREAD_GRAYSCALE)

print("=== Okuma Modu Karşılaştırması ===")
print(f"IMREAD_UNCHANGED (renkli): shape={renkli.shape}, dtype={renkli.dtype}, ")
print(f"   RAM: {renkli.nbytes/1024:.1f} KB")
print(f"IMREAD_UNCHANGED (saydam): shape={saydam.shape}, dtype={saydam.dtype}, ")
print(f"   RAM: {saydam.nbytes/1024:.1f} KB")
print(f"IMREAD_GRAYSCALE          : shape={gri.shape}, dtype={gri.dtype}, ")
print(f"   RAM: {gri.nbytes/1024:.1f} KB")
print(f"\nGri görüntü renkli görüntünün {renkli.nbytes/gri.nbytes:.1f}x daha az yer kaplar!")

cv.imshow("gri tonlamali resim", gri)
cv.waitKey(0)
cv.destroyAllWindows()

## 2.2 Üç Görüntüyü Arka Arkaya Gösterme

`waitKey(n)` ve `destroyWindow()` kombinasyonu ile slayt gösterisi:

In [ ]:
import cv2 as cv
import numpy as np

renkli = cv.imread("resim/manzara.jpg", cv.IMREAD_UNCHANGED)
saydam = cv.imread("resim/bilgisayar.png", cv.IMREAD_UNCHANGED)
gri    = cv.imread("resim/manzara.jpg", cv.IMREAD_GRAYSCALE)

print(renkli.shape)
print(saydam.shape)
print(gri.shape)

# Her birini 3 saniye göster, sonra kapat
cv.imshow("1", renkli)
cv.waitKey(3000)
cv.destroyWindow("1")

cv.imshow("2", saydam)
cv.waitKey(3000)
cv.destroyWindow("2")

cv.imshow("3", gri)
cv.waitKey(3000)
cv.destroyAllWindows()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>waitKey() dönüş değerini kullanmak:</b><br><code>tus = cv.waitKey(0)</code> basılan tuşun ASCII kodunu döndürür.<br><code>if tus == ord("q"): break</code> → q tuşuna basınca çık<br><code>if tus == 27: break</code> → ESC tuşuna basınca çık<br><code>if tus == ord("s"): cv.imwrite(...)</code> → s tuşuna basınca kaydet<br><br>Bu pattern, video oynatmada temel kontrol mekanizmasıdır. <code>waitKey(1)</code> 1ms bekler → neredeyse hemen devam eder (30+ FPS video için ideal).</span></div>

---
# 💾 BÖLÜM 3: Görüntü Kaydetme — `cv.imwrite()`

## 3.1 Temel Kaydetme

NumPy dizisinden görüntü oluşturup kaydetmek:

In [ ]:
import cv2 as cv
import numpy as np

# Sıfırdan gradient görüntü oluştur
dizi = np.zeros((256, 256), dtype="uint8")

# Her sütunu o sütunun indeksiyle doldur → yatay gradient
for i in range(0, 255):
    dizi[:, i] = i   # tüm satırlar, i. sütun = i değeri

print(f"Oluşturulan dizi: shape={dizi.shape}, dtype={dizi.dtype}")
print(f"Minimum değer: {dizi.min()}, Maksimum: {dizi.max()}")

# JPEG olarak kaydet
cv.imwrite("resim/uretilmis.jpg", dizi)

# Kaydedilen dosyayı tekrar oku ve göster
uretilmis_resim = cv.imread("resim/uretilmis.jpg")
cv.imshow("1", uretilmis_resim)
cv.waitKey(0)
cv.destroyAllWindows()
print("Gradient görüntü başarıyla kaydedildi.")

## 3.2 Farklı Formatlarda Kaydetme

Aynı görüntüyü JPG, BMP ve PNG olarak kaydedip dosya boyutlarını karşılaştıralım:

In [ ]:
import cv2 as cv
import numpy as np
import os

dizi = np.zeros((256, 256), dtype="uint8")
for i in range(0, 255):
    dizi[:, i] = i

# Üç formatta kaydet
cv.imwrite("resim/uretilmis.jpg", dizi)
cv.imwrite("resim/uretilmis.bmp", dizi)
cv.imwrite("resim/uretilmis.png", dizi)

# Dosya boyutlarını karşılaştır
print("Format Karşılaştırması:")
print("-" * 45)
for fmt in ["jpg", "bmp", "png"]:
    yol = f"resim/uretilmis.{fmt}"
    boyut = os.path.getsize(yol)
    print(f"{fmt.upper():4}: {boyut:8,} byte  ({boyut/1024:.1f} KB)")

ham_boyut = dizi.nbytes
print(f"\nHam numpy boyutu: {ham_boyut:,} byte ({ham_boyut/1024:.1f} KB)")
print(f"JPEG sıkıştırma oranı: {ham_boyut/os.path.getsize('resim/uretilmis.jpg'):.1f}x")
print(f"PNG  sıkıştırma oranı: {ham_boyut/os.path.getsize('resim/uretilmis.png'):.1f}x")
print(f"BMP  sıkıştırma oranı: {ham_boyut/os.path.getsize('resim/uretilmis.bmp'):.1f}x")

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 JPEG Sıkıştırmanın Matematiği</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">JPEG sıkıştırma 3 ana adımdan oluşur:<br><br><b>1. Renk Uzayı Dönüşümü (BGR → YCbCr):</b> İnsan gözü renk farkına (Cb, Cr) kıyasla parlaklığa (Y) çok daha duyarlıdır. Renk kanalları daha agresif sıkıştırılabilir.<br><br><b>2. DCT (Ayrık Kosinüs Dönüşümü):</b> Her 8×8 piksel blok, frekans bileşenlerine ayrıştırılır. Düşük frekanslar (genel şekil) korunur, yüksek frekanslar (ince detay) atılır.<br><br><b>3. Huffman Kodlama:</b> DCT katsayıları lossless sıkıştırılır.<br><br>JPEG 1986'da Joint Photographic Experts Group tarafından geliştirildi. İnternetin fotoğrafla dolmasını mümkün kılan algoritmadır — 1990'larda 28.8k modem ile yüksek kaliteli fotoğraf indirilebilmesini sağladı.</span></div>

## 3.3 Kalite Parametresi ile JPEG Kaydetme

`IMWRITE_JPEG_QUALITY` parametresi 0–100 arasında kalite belirler. Varsayılan: 95. Her kalitede dosya boyutu ve görsel kalite arasındaki dengeyi inceleyelim:

In [ ]:
import cv2 as cv
import numpy as np
import os

gri = cv.imread("resim/manzara.jpg", cv.IMREAD_GRAYSCALE)

print("JPEG Kalite Karşılaştırması:")
print("-" * 55)
baslik = f"{"Kalite":>8} | {"Boyut (KB)":>12} | {"Sikistirma":>12}"
print(baslik)
print("-" * 55)

for kalite in [10, 30, 50, 70, 85, 95, 100]:
    cv.imwrite(f"resim/kalite_{kalite}.jpg", gri,
               [cv.IMWRITE_JPEG_QUALITY, kalite])
    boyut = os.path.getsize(f"resim/kalite_{kalite}.jpg") / 1024
    oran = gri.nbytes / (boyut * 1024)
    print(f"{kalite:>8} | {boyut:>10.1f}  | {oran:>10.1f}x")

# Varsayılan kalite (95) ile göster
cv.imwrite("resim/manzara_gri.jpg", gri, [cv.IMWRITE_JPEG_QUALITY, 70])
manzara_gri = cv.imread("resim/manzara_gri.jpg")
print(manzara_gri)
cv.imshow("gri tonlamali resim", manzara_gri)
cv.waitKey(0)
cv.destroyAllWindows()

## 3.4 Tuşa Basınca Kaydetme — İnteraktif Kontrol

In [ ]:
import cv2 as cv
import numpy as np

# Gradient görüntüsü
dizi = np.zeros((256, 256), dtype="uint8")
for i in range(0, 255):
    dizi[:, i] = i

# Önce kaydedip göster (demo amaçlı: önceden var olanı oku)
cv.imwrite("resim/uretilmis.jpg", dizi)
uretilmis_resim = cv.imread("resim/uretilmis.jpg")
cv.imshow("1", uretilmis_resim)

print("Tuş Kılavuzu:")
print("  j → JPEG olarak kaydet")
print("  p → PNG  olarak kaydet")
print("  t → TIFF olarak kaydet")
print("  Diğer → Tekrar dene mesajı")

while True:
    tus = cv.waitKey(0)
    print(f"Basılan tuş kodu: {tus} (karakter: {chr(tus) if 32<=tus<127 else '?'} )")

    if tus == ord("j"):
        cv.imwrite("resim/uretilmis_tusla.jpg", dizi)
        print("✓ JPEG kaydedildi")
        break
    elif tus == ord("p"):
        cv.imwrite("resim/uretilmis_tusla.png", dizi)
        print("✓ PNG kaydedildi")
        break
    elif tus == ord("t"):
        cv.imwrite("resim/uretilmis_tusla.tiff", dizi)
        print("✓ TIFF kaydedildi")
        break
    else:
        print("Hemşerim bir daha dene (j/p/t)")

cv.destroyAllWindows()

<div style="background: #2d2200; border-left: 5px solid #f9c74f; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f9c74f; font-size: 1.08em;">⚠️ Kritik Nokta</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>waitKey() ve `& 0xFF` neden gerekebilir?</b><br><br>Bazı sistemlerde (özellikle Linux 64-bit) <code>cv.waitKey()</code> 32-bit bir değer döndürür ama yalnızca ilk 8 bit tuş kodudur. <code>tus = cv.waitKey(0) & 0xFF</code> yalnızca düşük 8 biti alır.<br><br>Karşılaştırma: <code>if tus == ord("q")</code> yerine <code>if tus & 0xFF == ord("q")</code> yazmak daha güvenlidir. Bazı klavye layout'larında Türkçe karakterler de farklı kodlar üretebilir.</span></div>

---
# 🎨 BÖLÜM 4: Programatik Görüntü Üretme

## 4.1 Büyük Gradient Görüntüsü

1024×1024 piksellik gradient oluşturalım ve farklı formatlarda kaydedelim:

In [ ]:
import cv2 as cv
import numpy as np
import os

# 1024×1024 gradient
dizi = np.zeros((1024, 1024), dtype="uint8")

# Yalnızca 0-254 arası (255 son sütuna ayrılmış)
for i in range(0, 255):
    dizi[:, i] = i

print(f"Dizi boyutu: {dizi.shape}, RAM: {dizi.nbytes/1024:.0f} KB")

# Üç formatta kaydet
for fmt in ["jpg", "bmp", "png"]:
    yol = f"resim/uretilmis.{fmt}"
    cv.imwrite(yol, dizi)
    boyut = os.path.getsize(yol)
    print(f"{fmt.upper():4}: {boyut:,} byte ({boyut/1024:.1f} KB)")

uretilmis_resim = cv.imread("resim/uretilmis.jpg")
cv.imshow("1", uretilmis_resim)
cv.waitKey(0)
cv.destroyAllWindows()

## 4.2 Simetrik Gradient — İleri Geri

Ortada beyaz (255) olan bir "dağ" profili:

In [ ]:
import cv2 as cv
import numpy as np

dizi = np.zeros((512, 512), dtype="uint8")

# Üst ve alt yarılarda simetrik gradient
for i in range(0, 256):
    dizi[i,   :256] = i        # sol üst: 0→255
    dizi[-i,  :256] = i        # sol alt: 0→255 (aşağıdan yukarı)
    dizi[i,  256:512] = i      # sağ üst
    dizi[-i, 256:512] = i      # sağ alt

dizi[256, 0:256]   = 255       # üst yarı orta çizgisi: tam beyaz
dizi[256, 256:512] = 255       # alt yarı orta çizgisi: tam beyaz

print(f"Dizi: {dizi.shape}, min={dizi.min()}, max={dizi.max()}")

cv.imwrite("resim/uretilmis.jpg", dizi)
uretilmis_resim = cv.imread("resim/uretilmis.jpg")
cv.imshow("1", uretilmis_resim)
cv.waitKey(0)
cv.destroyAllWindows()

---
# 🌈 BÖLÜM 5: Renk Gradient Görüntüleri

## 5.1 Siyahtan Beyaza (Yukarıdan Aşağıya)

Satır indeksini kullanarak dikey gradient:

In [ ]:
import cv2 as cv
import numpy as np

# Gri tonlamalı dikey gradient
resim = np.zeros((256, 256), dtype="uint8")

for i in range(256):
    resim[i, :] = i   # i. satır tamamı = i değeri

cv.imshow("Siyahtan Beyaza (dikey)", resim)
cv.waitKey(0)
cv.destroyAllWindows()

## 5.2 Maviden Kırmızıya — Renkli Yatay Gradient

In [ ]:
import cv2 as cv
import numpy as np

# 256×256 renkli görüntü: 3 kanal (BGR)
dizi = np.zeros((256, 256, 3), dtype="uint8")

print(f"shape: {dizi.shape}, ndim: {dizi.ndim}, dtype: {dizi.dtype}")
print(f"İlk birkaç piksel:\n{dizi[:2, :4]}")

# Sütun bazlı renk atama:
# [255-i, 127, i] = [Mavi azalan, Yeşil sabit, Kırmızı artan]
# i=0:   [255, 127, 0]   → tam mavi
# i=127: [128, 127, 127] → gri/mor geçiş
# i=255: [0,   127, 255] → tam kırmızı (OpenCV BGR)
for i in range(0, 255):
    dizi[:, i] = [255 - i, 127, i]

print(f"\nSon hali: min={dizi.min()}, max={dizi.max()}")
cv.imshow("1", dizi)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>BGR ile renk karıştırma:</b><br><code>[255-i, 127, i]</code>: Mavi ve kırmızı karşılıklı değişirken yeşil ortada sabit.<br>Bu, renk tekerleğinin bir diliminden geçiş yapar: Mavi → Mor → Kırmızı.<br><br>Farklı renk geçişleri için:<br><code>[0, 255-i, i]</code> → Yeşil'den Kırmızı'ya<br><code>[i, 0, 255-i]</code> → Kırmızı'dan Mavi'ye<br><code>[255-i, i, 0]</code> → Mavi'den Yeşil'e</span></div>

## 5.3 Maviden Yeşile, Yeşilden Kırmızıya (Dikey — 512×512)

In [ ]:
import cv2 as cv
import numpy as np

dizi = np.zeros((512, 512, 3), dtype="uint8")
print(f"shape: {dizi.shape}, ndim: {dizi.ndim}, dtype: {dizi.dtype}")

# Üst yarı (satır 0-254): Mavi → Yeşil
# Alt yarı (satır 255-511): Yeşil → Kırmızı
for i in range(0, 255):
    dizi[i,      :] = [255 - i, i, 0]   # [Mavi↓, Yeşil↑, 0]  BGR: mavi→yeşil
    dizi[i + 255,:] = [0, 255 - i, i]   # [0, Yeşil↓, Kırmızı↑]  yeşil→kırmızı

cv.imshow("1", dizi)
cv.waitKey(0)
cv.destroyAllWindows()

## 5.4 Örnek Sorular — Küçük Alıştırmalar

In [ ]:
import cv2 as cv
import numpy as np

# SORU: Kırmızıdan (0,0,255) Mavi'ye (255,0,0) geçen
# yatay gradient oluşturun
dizi = np.zeros((256, 256, 3), dtype="uint8")
for i in range(256):
    dizi[:, i] = [i, 0, 255 - i]   # BGR: [Mavi↑, 0, Kırmızı↓]

cv.imshow("Kırmızı→Mavi", dizi)
cv.waitKey(0)
cv.destroyAllWindows()

---
# ♟️ BÖLÜM 6: Satranç Tahtası Oluşturma

## 6.1 Temel Mantık

Satranç tahtasında bir kare `(i, j)` konumundaki karenin rengi `(i+j) % 2` ile belirlenir.

```
i=0, j=0: (0+0)%2 = 0 → Beyaz
i=0, j=1: (0+1)%2 = 1 → Siyah
i=1, j=0: (1+0)%2 = 1 → Siyah
i=1, j=1: (1+1)%2 = 0 → Beyaz
```

Her kare 100×100 piksel, toplam tahta 800×800 piksel.

In [ ]:
import cv2 as cv
import numpy as np

dizi = np.zeros((800, 800), dtype="uint8")

for i in range(0, 8):
    for j in range(0, 8):
        if (i + j) % 2 == 0:
            # Sol üst köşe: (j*100, i*100)
            # Sağ alt köşe: (j*100+100, i*100+100)
            dizi[i*100 : i*100+100, j*100 : j*100+100] = 255

print(f"Satranç tahtası: {dizi.shape}")
cv.imshow("1", dizi)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 Satranç ve Görüntü İşleme İlişkisi</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Satranç tahtası, bilgisayar görüsünde son derece önemli bir kalibrasyon aracıdır! <b>Kamera kalibrasyonunda</b> (cv.calibrateCamera) standart referans nesnesi satranç tahtasıdır. Nedeni: köşe noktaları (chess corners) piksel-düzeyinde tespit edilebilir, gerçek dünya koordinatları tam bilinir.<br><br>cv.findChessboardCorners() fonksiyonu bu amaç için tasarlanmıştır: bir satranç tahtasının iç köşelerini alt-piksel hassasiyetiyle bulur. Bu, lens distorsiyonunu, kamera matrisini ve dış parametreleri hesaplamak için temel adımdır.<br><br>Otonom araçlar, endüstriyel robotlar ve 3D tarayıcılar hep bu satranç tahtası kalibrasyonuyla başlar!</span></div>

## 6.2 Satranç Tahtasına Manzara Resmi Yerleştirme

In [ ]:
import cv2 as cv
import numpy as np
import random

dizi = np.zeros((800, 800, 3), dtype="uint8")
manzara = cv.imread("resim/manzara.jpg")

for i in range(0, 8):
    for j in range(0, 8):
        if (i + j) % 2 == 0:  # yalnızca beyaz karelere
            # Manzaradan rastgele 100×100 bölge seç
            x = random.randint(0, manzara.shape[0] - 100)
            y = random.randint(0, manzara.shape[1] - 100)

            # Satranç tahtasındaki ilgili kareye yerleştir
            dizi[i*100 : i*100+100, j*100 : j*100+100] = manzara[x:x+100, y:y+100]

cv.imshow("1", dizi)
cv.waitKey(0)
cv.destroyAllWindows()

## 6.3 Animasyonlu Satranç Tahtası — Rastgele Sırayla

In [ ]:
import cv2 as cv
import numpy as np
import random

chess_board = np.zeros((8*64, 8*64, 3), dtype=np.uint8)
manzara = cv.imread("resim/manzara.jpg")

# Beyaz kareleri listeye al
white_cells = []
for i in range(8):
    for j in range(8):
        if (i + j) % 2 == 0:
            white_cells.append((i, j))

# Listeyi karıştır — rastgele sıra
random.shuffle(white_cells)

# Kareleri tek tek animasyonlu olarak doldur
for (i, j) in white_cells:
    x = random.randint(0, manzara.shape[0] - 64)
    y = random.randint(0, manzara.shape[1] - 64)

    top_left     = (i * 64, j * 64)
    bottom_right = ((i + 1) * 64, (j + 1) * 64)

    selected_area = manzara[x:x + 64, y:y + 64]
    selected_area_resized = cv.resize(
        selected_area,
        (bottom_right[1] - top_left[1], bottom_right[0] - top_left[0])
    )

    chess_board[top_left[0]:bottom_right[0], top_left[1]:bottom_right[1]] = selected_area_resized

    cv.imshow("Chess Board", chess_board)
    cv.waitKey(100)   # 100ms bekle → animasyon etkisi

cv.waitKey(0)
cv.destroyAllWindows()

---
# 🎬 BÖLÜM 7: Video Okuma — `cv.VideoCapture`

## 7.1 Video Nedir? — Teknik Bakış

Video, art arda gösterilen görüntüler dizisidir. Her görüntü bir **kare (frame)** olarak adlandırılır.

```
Video:
  Kare 1  Kare 2  Kare 3  ...  Kare N
  ┌────┐  ┌────┐  ┌────┐       ┌────┐
  │    │  │    │  │    │  ...  │    │
  └────┘  └────┘  └────┘       └────┘
     |                              |
     └── 1/FPS saniye aralıkla ─────┘

30 FPS video: saniyede 30 kare
60 FPS video: saniyede 60 kare (oyun/spor)
120 FPS video: yüksek hız çekim
```

| Özellik | Metod | Açıklama |
|---------|-------|----------|
| FPS | `video.get(cv.CAP_PROP_FPS)` | Kare/saniye |
| Genişlik | `video.get(cv.CAP_PROP_FRAME_WIDTH)` | Piksel cinsinden |
| Yükseklik | `video.get(cv.CAP_PROP_FRAME_HEIGHT)` | Piksel cinsinden |
| Kare sayısı | `video.get(cv.CAP_PROP_FRAME_COUNT)` | Toplam kare |
| Süre | `kare_sayisi / fps` | Saniye cinsinden |

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Video Sıkıştırmanın Tarihi:</b><br><br>1990'larda sıkıştırılmamış video devasa boyuttaydı. 720p 30FPS video → 720×1280×3×30 = 83 MB/saniye. 2 saatlik film = 600 GB!<br><br><b>MPEG-1 (1993):</b> İlk yaygın video sıkıştırma standardı. VCD formatının temeli.<br><b>MPEG-2 (1995):</b> DVD ve dijital televizyon. Hâlâ Blu-ray'de kullanılıyor.<br><b>H.264/AVC (2003):</b> YouTube, Netflix, Blu-ray. MPEG-2'den 2× daha verimli.<br><b>H.265/HEVC (2013):</b> 4K streaming. H.264'ten 2× daha verimli.<br><b>AV1 (2018):</b> Google/Netflix açık kaynak. H.265'ten %30 daha verimli.<br><br>OpenCV'nin VideoCapture'ı bu codec'lerin hepsini FFmpeg aracılığıyla okuyabilir.</span></div>

In [ ]:
import cv2 as cv

video = cv.VideoCapture("video/ornek.mp4")

# Video bilgilerini oku
fps   = video.get(cv.CAP_PROP_FPS)
en    = video.get(cv.CAP_PROP_FRAME_WIDTH)
boy   = video.get(cv.CAP_PROP_FRAME_HEIGHT)
kare  = video.get(cv.CAP_PROP_FRAME_COUNT)

print(f"FPS:         {fps:.1f}")
print(f"Çözünürlük:  {int(en)}×{int(boy)} piksel")
print(f"Kare sayısı: {int(kare)}")
print(f"Süre:        {kare/fps:.1f} saniye ({kare/fps/60:.1f} dakika)")
print(f"Ham boyut:   {en*boy*3*kare/1024/1024:.0f} MB (sıkıştırmasız)")

while video.isOpened():
    ret, frame = video.read()
    print(ret)
    if ret:
        cv.imshow("video", frame)
        if cv.waitKey(33) == 27:   # 33ms ≈ 30fps, ESC=27
            break

video.release()
cv.destroyAllWindows()

## 7.2 Video Oynatma — Yavaştan Hızlıya + Aynalama

Örnek video: her kare biraz daha hızlı oynatılır + yatay ayna efekti:

In [ ]:
import cv2 as cv
import numpy as np

video = cv.VideoCapture("video/ornek.mp4")

fps = video.get(cv.CAP_PROP_FPS)
en  = video.get(cv.CAP_PROP_FRAME_WIDTH)
boy = video.get(cv.CAP_PROP_FRAME_HEIGHT)
print(f"{fps} FPS, {int(en)}×{int(boy)}")

hiz = 100           # başlangıç bekleme ms
hizlandirma_sayac = 0

while video.isOpened():
    ret, frame = video.read()
    if not ret:
        break

    # Yatay aynalama — Çözüm 1: sütunları ters sırala
    # frame2 = np.zeros((360, 640, 3), dtype="uint8")
    # for i in range(0, 640):
    #     frame2[:, i] = frame[:, 639-i]

    # Yatay aynalama — Çözüm 2: NumPy dilimleme (çok daha hızlı!)
    frame2 = frame[:, ::-1]   # tüm satırlar, sütunlar tersten

    cv.imshow("video", frame2)

    # Hızlandırma: her 2 karede 1ms azalt
    if hiz > 10:
        hizlandirma_sayac += 1
        if hizlandirma_sayac % 2 == 0:
            hiz -= 1

    if cv.waitKey(hiz) == 27:
        break

video.release()
cv.destroyAllWindows()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>frame[:, ::-1] nasıl çalışır?</b><br><br>NumPy dilimleme sözdizimi: <code>dizi[satir_dilimi, sutun_dilimi]</code><br><code>:</code> → tüm satırlar<br><code>::-1</code> → tüm sütunlar, ama tersten (son sütundan ilk sütuna)<br><br>Bu, bellekte yeni bir kopya oluşturmaz — bir <b>view</b> döndürür. Yani orijinal veriyi tersine "gösteren" sanal bir penceredir. Bellekte ekstra veri taşımak gerekmez — bu yüzden çok hızlıdır.<br><br><code>frame[::-1, :]</code> → dikey flip (satırlar ters)<br><code>frame[::-1, ::-1]</code> → 180° dönüş<br><code>frame[:, ::-1, :]</code> → BGR → RGB çevirme (!)</span></div>

---
# 🖌️ BÖLÜM 8: Gelişmiş Görüntü Üretimi

## 8.1 Renk Paleti — HSV Uzayında

HSV (Hue-Saturation-Value) renk uzayı, renkleri insan algısına daha yakın temsil eder. Paint uygulamalarındaki renk seçici bu mantıkla çalışır:

In [ ]:
"""
Paint'teki gibi renk paleti oluşturun
"""
import cv2 as cv
import numpy as np

# Piksel piksel HSV değeri atama
image = np.zeros((256, 256, 3), dtype=np.uint8)

for i in range(256):
    for s in range(256):
        # Hue: sütun indeksi (0-255 → tüm renk çemberi)
        # Saturation: satırın tersi (üstte düşük doygunluk=beyaz, altta yüksek=canlı renk)
        # Value: sabit 255 (tam parlak)
        image[i, s] = [s, 255 - i, 255]

# HSV → BGR dönüşümü ile görüntüle
bgr_image = cv.cvtColor(image, cv.COLOR_HSV2BGR)

cv.imshow("HSV Renk Paleti", bgr_image)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 HSV Renk Uzayı Neden Var?</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>RGB'nin sorunu:</b> "Kırmızının daha açığını istiyorum" demek için üç kanalı da aynı anda değiştirmek gerekir. Bu, sezgisel değildir.<br><br><b>HSV'nin çözümü:</b><br>• <b>H (Hue — Ton):</b> Renk çemberi 0°-360° → hangi renk? (Kırmızı=0°, Yeşil=120°, Mavi=240°)<br>• <b>S (Saturation — Doygunluk):</b> 0=gri, 255=canlı renk<br>• <b>V (Value — Parlaklık):</b> 0=siyah, 255=en parlak<br><br>Renk takibinde (örn. kırmızı nesne bul) HSV çok daha güçlüdür: <code>cv.inRange(hsv, [160,100,100], [179,255,255])</code> ile kırmızı pikselleri aydınlatma koşullarından bağımsız bulabilirsiniz. RGB ile aynı işlem hem R hem G hem B aralığı kontrolü gerektirir.</span></div>

In [ ]:
"""
Copilot'a yaptır: gelişmiş BGR renk paleti
"""
import cv2
import numpy as np

genislik = 512
yukseklik = 512
red_degeri = 128

# Linspace: 0-255 arası eşit aralıklı değerler
x = np.linspace(0, 255, genislik, dtype=np.uint8)
y = np.linspace(0, 255, yukseklik, dtype=np.uint8)

# tile: tek boyutu çoğaltarak 2D diziye çevir
green_kanali = np.tile(x, (yukseklik, 1))             # her satır aynı x değerleri
blue_kanali  = np.tile(y.reshape(-1, 1), (1, genislik)) # her sütun aynı y değerleri
red_kanali   = np.full((yukseklik, genislik), red_degeri, dtype=np.uint8)

# dstack: derinlik ekseninde birleştir → (H, W, 3)
palette = np.dstack((blue_kanali, green_kanali, red_kanali))

def mouse_callback(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        b, g, r = palette[y, x]
        print(f"Tıklanan piksel ({x},{y}) → BGR:({b},{g},{r})  RGB:({r},{g},{b})")

cv2.namedWindow("BGR Renk Paleti")
cv2.setMouseCallback("BGR Renk Paleti", mouse_callback)
cv2.imshow("BGR Renk Paleti", palette)
cv2.waitKey(0)
cv2.destroyAllWindows()

## 8.2 NumPy ile Vektörize Renk Üretimi — Hız Karşılaştırması

Döngü vs NumPy vektörizasyonu arasındaki farkı somutlayalım:

In [ ]:
import numpy as np
import time

BOYUT = 512

# Yöntem 1: Python döngüsü (yavaş)
baslangic = time.time()
dizi_dongu = np.zeros((BOYUT, BOYUT, 3), dtype=np.uint8)
for i in range(BOYUT):
    for j in range(BOYUT):
        dizi_dongu[i, j] = [j % 256, i % 256, (i+j) % 256]
dongu_sure = time.time() - baslangic
print(f"Döngü yöntemi:      {dongu_sure:.3f} saniye")

# Yöntem 2: NumPy vektörizasyonu (hızlı)
baslangic = time.time()
# meshgrid: her piksel için satır ve sütun değeri
rows, cols = np.mgrid[0:BOYUT, 0:BOYUT]
dizi_numpy = np.zeros((BOYUT, BOYUT, 3), dtype=np.uint8)
dizi_numpy[:,:,0] = cols % 256    # Blue kanal = sütun değeri
dizi_numpy[:,:,1] = rows % 256    # Green kanal = satır değeri
dizi_numpy[:,:,2] = (rows + cols) % 256  # Red kanal
numpy_sure = time.time() - baslangic
print(f"NumPy vektör:       {numpy_sure:.4f} saniye")
print(f"Hız farkı:          {dongu_sure/numpy_sure:.0f}x daha hızlı!")

# Sonuçların aynı olduğunu doğrula
print(f"Sonuçlar eşit mi?   {np.array_equal(dizi_dongu, dizi_numpy)}")

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">🔄 Karşılaştırma</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Neden NumPy bu kadar hızlı?</b><br><br>Python döngüsü: Her iterasyonda Python interpreter devreye girer, nesne oluşturulur, tip kontrolü yapılır → ağır overhead.<br><br>NumPy: C ile yazılmış BLAS/LAPACK kütüphaneleri kullanır. Tüm veriye tek seferde işlem uygulanır (SIMD komutları). Python interpreter yalnızca bir kez çağrılır.<br><br>Gerçek sayı: 512×512 piksel işleme için Python döngüsü ~2-5 saniye, NumPy ~0.001 saniye → <b>1000-5000x hız farkı</b>. Bu yüzden bilgisayarlı görüde "asla ham Python döngüsü kullanma" kuralı var.</span></div>

---
# 🏛️ BÖLÜM 9: Üniversite Seviyesi Alıştırmalar

Aşağıdaki sorular bu haftanın konularını bilgisayar bilimi, matematik ve mühendislik perspektifiyle birleştirmenizi gerektirir. **Cevaplar verilmemiştir.**

---
## ❓ Soru 1 — Fraktal Görüntü Üretici

**Zorluk: ⭐⭐⭐☆☆**

Mandelbrot kümesini piksel piksel hesaplayarak 512×512 gri tonlamalı görüntü olarak üreten ve kaydeden bir program yazın.

### Görev:
1. Her piksel `(px, py)` için karmaşık sayı üret: `c = (px/512)*3.5 - 2.5 + ((py/512)*2 - 1)j`
2. `z = 0` başlayarak `z = z² + c` iterasyonunu uygula
3. Kaçış iterasyon sayısı `n` piksel parlaklığını belirlesin
4. Maksimum iterasyon sayısını 100, 500, 1000 için karşılaştır
5. Sonucu hem gri hem sahte renkli (colormap) olarak kaydet

### Düşünmeniz Gereken Sorular:
- NumPy vektörizasyonu ile döngü arasında kaç kat hız farkı oluşur?
- Fraktallar neden sonsuz ayrıntı içerir ama sonlu bellekte saklanabilir?
- "Kaçış hızı boyama" (escape velocity coloring) neden ince bantlar oluşturur?

```python
# Başlangıç iskeleti
import numpy as np
import cv2 as cv

def mandelbrot(px, py, max_iter=100):
    # c noktasını hesapla, iterasyonu uygula
    # Kaçış iterasyon sayısını döndür
    pass

def goruntu_uret(boyut=512, max_iter=100):
    # Tüm pikseller için mandelbrot() çağır
    # dtype=uint8 görüntü döndür
    pass
```

---
## ❓ Soru 2 — Fourier Analizi ile Görüntü Spektrumu

**Zorluk: ⭐⭐⭐⭐☆**

Bir görüntünün 2B Fourier dönüşümünü hesaplayın, görselleştirin ve frekans filtresi uygulayarak görüntüyü geri kurtarın.

### Görev:
1. `np.fft.fft2()` ile görüntünün 2B FFT'sini hesaplayın
2. `np.fft.fftshift()` ile düşük frekansları merkeze taşıyın
3. Güç spektrumunu `20*log10(|F|)` ile görselleştirin
4. Alçak geçiren filtre (low-pass): merkezdeki bir daireyi tut, geri kalanı sıfırla
5. Yüksek geçiren filtre (high-pass): merkezi sıfırla (yalnızca kenarları tut)
6. `ifft2()` ile filtrelenmiş görüntüyü geri oluşturun

### Düşünmeniz Gereken Sorular:
- Low-pass filtre uygulanmış görüntü nasıl görünür? Neden?
- High-pass filtre görüntüden ne çıkartır? Hangi bilgi bilgisayarlı görü için değerlidir?
- JPEG sıkıştırma ile FFT filtresi arasındaki kavramsal benzerlik nedir?

```python
# Başlangıç iskeleti
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

def fourier_analiz(goruntu_yolu):
    img = cv.imread(goruntu_yolu, cv.IMREAD_GRAYSCALE)
    # 1. FFT hesapla
    # 2. Spektrumu görselleştir
    # 3. Low/high-pass filtre uygula
    # 4. Geri kurtarılan görüntüyü göster
    pass
```

---
## ❓ Soru 3 — Gerçek Zamanlı Kamera Histogram Analizörü

**Zorluk: ⭐⭐⭐☆☆**

Kameradan gelen her kareyi gerçek zamanlı olarak analiz eden, histogram ve temel istatistikleri yan yana gösteren interaktif bir araç yazın.

### Görev:
1. Kameradan sürekli kare oku
2. Her kare için BGR kanallarının histogramını `cv.calcHist()` ile hesapla
3. Histogram görüntüsünü `cv.polylines()` veya `cv.line()` ile çiz (matplotlib kullanma!)
4. İstatistikleri kamera görüntüsüne `cv.putText()` ile yaz:
   - Ortalama parlaklık
   - Standart sapma
   - En yaygın renk (mode)
5. Kamera görüntüsü + histogram = yan yana `np.hstack()` ile birleştir
6. `h` tuşuna basınca histogram eşitleme (equalization) aç/kapat

### Düşünmeniz Gereken Sorular:
- Histogram eşitleme (equalization) görüntüyü nasıl değiştirir? Hangi tür görüntülerde etkilidir?
- CLAHE (Contrast Limited Adaptive Histogram Equalization) neden düz eşitten daha iyidir?
- Histogram istatistikleri yalnızca global bilgi verir — lokal özellikler nasıl yakalanır?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def histogram_gorsellestir(frame, h=200, w=256):
    # Her BGR kanalı için histogram hesapla ve görüntüye çiz
    # Döndür: (h, w, 3) BGR histogram görüntüsü
    pass

video = cv.VideoCapture(0)
eq_aktif = False
while video.isOpened():
    ret, frame = video.read()
    # TODO: histogramı hesapla, birleştir, göster
    pass
```

---
## ❓ Soru 4 — Steganografi: Görüntü İçine Gizli Mesaj

**Zorluk: ⭐⭐⭐⭐☆**

LSB (Least Significant Bit) steganografisi: bir metin mesajını görüntünün piksel değerlerine göze görünmez biçimde gömün, sonra geri çıkarın.

### Görev:
**Gömme (encode):**
1. Gizlenecek metni binary'e çevir (ASCII → 8-bit)
2. Her pikselin mavi kanalının son bitini (LSB) mesajın bir bitiyle değiştir
3. Sonunda özel bir "bitiş işaretçisi" ekle (örn. 8 sıfır bit)
4. Değiştirilmiş görüntüyü PNG olarak kaydet (JPEG mesajı bozar!)

**Çıkarma (decode):**
1. Her pikselin mavi kanalının son bitini oku
2. 8 bit grupla, ASCII karakterine çevir
3. Bitiş işaretçisine kadar oku, metni döndür

### Düşünmeniz Gereken Sorular:
- LSB değişikliği görsel fark yaratır mı? PSNR değeri ne olur?
- 512×512 görüntüye maksimum kaç karakter gizlenebilir?
- Bu yöntem şifreleme midir? Değilse farkı nedir?
- Steganografi tespiti (steganalysis) nasıl yapılır? χ² testi ne işe yarar?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def metni_gom(goruntu, metin):
    """Metni görüntünün LSB'lerine göm, yeni görüntüyü döndür."""
    gizli = goruntu.copy()
    binary_metin = "".join(format(ord(c), "08b") for c in metin) + "00000000"
    # TODO
    return gizli

def metni_cikart(goruntu):
    """Görüntüdeki gizli metni çıkar."""
    # TODO
    pass
```

---
## ❓ Soru 5 — Optik İllüzyon Üreteci

**Zorluk: ⭐⭐⭐⭐⭐**

Sadece NumPy ve OpenCV kullanarak matematiksel olarak üretilen bir optik illüzyon koleksiyonu oluşturun.

### Görev — 3 farklı illüzyon:

**1. Hermann Izgarası:** Izgaranın kavşaklarında olmayan gri noktalar görünür.
- Düzenli beyaz çizgili siyah ızgara oluştur
- Bir döngü içinde siyah dikdörtgenler çizerek ızgara yapısını oluştur

**2. Dönen Çarklar (Fraser Spiral):**
- Biri diğerinin içine kademeli olarak yerleştirilmiş daireler
- `cv.ellipse()` ile iç içe geçmiş daireler, her biri biraz döndürülmüş

**3. Mach Bantları:**
- Kademeli parlaklık geçişinin yanındaki sarp kenarda insan gözü var olmayan koyu/açık bantlar "görür"
- Adımlı gradient oluştur: düz bölgeler + sarp geçiş

### Düşünmeniz Gereken Sorular:
- Hermann Izgarası neden gerçekmiş gibi görünür? (Lateral inhibition nedir?)
- İnsan görsel sistemindeki hangi mekanizma bu yanılsamalara yol açar?
- Mach bantları gerçekte görüntüde var mı? Konvolüsyon ile doğrulayın.
- Yapay sinir ağları da aynı optik yanılsamalara düşer mi? (Araştırın!)

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def hermann_izgara(boyut=512, kare_boyut=50, cizgi_kalinligi=10):
    """Hermann ızgarası illüzyonu üret."""
    pass

def mach_bantlari(boyut=512, adim_sayisi=8):
    """Mach bantları illüzyonu üret."""
    pass

def dongusel_carklar(boyut=512):
    """Döngüsel çark illüzyonu üret."""
    pass
```

---
<div style="background: linear-gradient(135deg, #050505 0%, #0d1117 40%, #0a2a1a 100%);padding: 38px 42px; border-radius: 18px; text-align: center; margin-top: 20px; border: 1px solid #2ea04344;"><h2 style="color: #2ea043; font-family: 'Courier New', monospace; margin: 0 0 22px 0;">🎯 5. Hafta Özeti</h2><div style="color: #cdd6f4; text-align: left; max-width: 700px; margin: 0 auto; line-height: 2.2;">✅ <b>Görüntü = NumPy dizisi:</b> uint8, shape=(H,W) veya (H,W,3), 0-255 arası piksel değerleri<br>✅ <b>cv.imwrite():</b> JPEG (kayıplı), PNG (kayıpsız), BMP (ham); IMWRITE_JPEG_QUALITY parametresi<br>✅ <b>İnteraktif kaydetme:</b> waitKey() + ord() ile tuş bazlı kontrol<br>✅ <b>Gradient üretimi:</b> döngüyle sütun/satır bazlı piksel atama<br>✅ <b>Renkli gradient:</b> BGR kanallarına farklı formüller → renk geçişleri<br>✅ <b>Satranç tahtası:</b> (i+j)%2 modular aritmetik ile kare rengi belirleme<br>✅ <b>HSV renk uzayı:</b> Hue/Saturation/Value ve cv.cvtColor()<br>✅ <b>Video okuma:</b> VideoCapture, CAP_PROP_FPS/WIDTH/HEIGHT/FRAME_COUNT<br>✅ <b>NumPy vektörizasyonu:</b> döngüden 1000x+ hızlı piksel işleme<br>✅ <b>uint8 taşma tuzağı:</b> cv.add()/subtract() vs + operatörü farkı</div><p style="color: #6b7280; margin: 22px 0 0 0; font-family: 'Courier New', monospace;">6. Hafta → Kamera, Video Kaydetme, Fare Olayları & Renk Takibi 🎥</p></div>